# Production ETL — At-Risk Student Dashboard

**Production-grade pipeline.** Real Moodle data only. Fails loud on any missing required field. Ready for industry partner deployment.

## Key differences from hybrid_etl

|                          | hybrid_etl                              | production_etl (this notebook)         |
|--------------------------|-----------------------------------------|----------------------------------------|
| Imports `etl/mocks.py`?  | YES                                     | **NO** (architecturally enforced)       |
| On token invalid         | Falls back to demo cohort                | **Raises `ProductionDataMissingError`**  |
| On empty cohort          | Falls back to demo cohort                | **Raises `ProductionDataMissingError`**  |
| On missing grade items   | Per-cell mock fill                       | **Raises** if all-empty; allows per-row gaps  |
| On `attendance` source   | Always simulated                         | **Raises** until Meshed integration is wired  |
| Output schema            | Same                                    | Same                                    |

## Notebook Purpose

- Production deployment on Microsoft Fabric (when migrated)
- Sanity check that real Moodle data is sufficient for the dashboard
- CI/scheduled runs where silent fallback would mask data quality issues

***NOT use this notebook*** 
- Development phase with sparse sandbox data — use `hybrid_etl.ipynb` instead


## 1. Setup

In [1]:
# =========================
# UNIVERSAL ENVIRONMENT SETUP + ETL IMPORTS
# =========================
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

import numpy as np     
import pandas as pd  

def load_env_safely():
    current_path = Path.cwd()
    for i in range(5):
        env_path = current_path / ".env"
        if env_path.exists():
            print(f"✅ .env found at: {env_path}")
            load_dotenv(env_path, override=True)
            return env_path
        current_path = current_path.parent
    print("❌ .env file NOT found!")
    return None

env_path = load_env_safely()

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Now safe to import etl modules — env vars are loaded
from etl import schemas as S
from etl import io as etl_io
from etl import moodle_client as mc
from etl import transforms as T
from etl import risk as R

print("\n===== ENV STATUS =====")
print("Working directory:", Path.cwd())
print("Base URL:", mc.MOODLE_BASE_URL)
print("Token exists:", bool(mc.MOODLE_TOKEN))
print("Token preview:", mc.MOODLE_TOKEN[:6] + "..." if mc.MOODLE_TOKEN else "EMPTY")
print("Endpoint:", mc.MOODLE_ENDPOINT)
print("======================\n")

✅ .env found at: /Users/Kay/MDS651 Industry Capstone/.env

===== ENV STATUS =====
Working directory: /Users/Kay/MDS651 Industry Capstone/Notebook
Base URL: https://spi.nsw.edu.au/learn
Token exists: True
Token preview: 4daf5d...
Endpoint: https://spi.nsw.edu.au/learn/webservice/rest/server.php



## 2. Configuration

In [2]:
DATA_DIR = PROJECT_ROOT.parent / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
STAR_SCHEMA_DIR = DATA_DIR / "star_schema"
etl_io.ensure_dirs(DATA_DIR, RAW_DIR, PROCESSED_DIR, STAR_SCHEMA_DIR)

# Minimum students required to consider the cohort meaningful for production.
# Below this, production ETL aborts rather than emit a near-empty dashboard.
MIN_STUDENT_COUNT = 5

# Whether attendance is available from a real source (e.g., Meshed integration).
# Set to True once a real attendance feed is wired into this notebook.
ATTENDANCE_SOURCE_AVAILABLE = False


## 3. Fail-loud helper

`ProductionDataMissingError` includes the exact remediation step so the operator (or industry partner) knows what to fix.

In [3]:
class ProductionDataMissingError(RuntimeError):
    """Raised when production ETL cannot proceed with real-only data."""

    def __init__(self, field: str, moodle_function: str | None, remediation: str):
        self.field = field
        self.moodle_function = moodle_function
        self.remediation = remediation
        msg = (
            f"Production data missing for field '{field}'\n"
            f"  Moodle function: {moodle_function or 'n/a'}\n"
            f"  Remediation:    {remediation}"
        )
        super().__init__(msg)


## 4. Pre-flight check (FAIL LOUD)

In [4]:
preflight = mc.preflight_check()

if not preflight["ok"]:
    failure_mode = preflight["failure_mode"]
    remediation_map = {
        "token_invalid": "Refresh Moodle Web Services token and update MOODLE_TOKEN in .env or environment.",
        "endpoint_blocked": "Contact Moodle admin to grant Web Services access to core_webservice_get_site_info.",
        "network_error": "Verify MOODLE_BASE_URL is reachable and there is no firewall/proxy issue.",
    }
    raise ProductionDataMissingError(
        field="site_info",
        moodle_function="core_webservice_get_site_info",
        remediation=remediation_map.get(failure_mode, "Investigate the Moodle API connection."),
    )

print("Pre-flight OK.")


[moodle] Pre-flight FAILED (token_invalid): invalidtoken: Invalid token - token not found


ProductionDataMissingError: Production data missing for field 'site_info'
  Moodle function: core_webservice_get_site_info
  Remediation:    Refresh Moodle Web Services token and update MOODLE_TOKEN in .env or environment.

## 5. Course catalog (FAIL LOUD)

In [ ]:
site_info_result = mc.extract_site_info()
courses_result = mc.extract_all_courses()
etl_io.save_json(site_info_result, RAW_DIR / "site_info.json")
etl_io.save_json(courses_result, RAW_DIR / "all_courses.json")

course_catalog = T.transform_course_catalog(courses_result, site_info_result)
site_df = T.transform_site_info(site_info_result)

if course_catalog.empty:
    raise ProductionDataMissingError(
        field="course_catalog",
        moodle_function="core_course_get_courses",
        remediation="Ensure the token has permission for course catalog. "
                    "Verify courses exist and are visible to the token's user.",
    )

print(f"Course catalog: {len(course_catalog)} courses")
display(course_catalog)


## 6. Real participant extraction (FAIL LOUD)

In [ ]:
participant_results = {}
for cid in course_catalog["course_id"].dropna().astype(int).tolist():
    result = mc.extract_course_participants(cid)
    participant_results[cid] = result
    etl_io.save_json(result, RAW_DIR / f"participants_course_{cid}.json")

real_participants_long = T.transform_course_participants(participant_results)

if real_participants_long.empty:
    raise ProductionDataMissingError(
        field="participants",
        moodle_function="core_enrol_get_enrolled_users",
        remediation="Verify the token has 'core_enrol_get_enrolled_users' permission "
                    "and that students are enrolled in at least one course.",
    )

# Filter to students for the cohort count
student_count = real_participants_long[
    real_participants_long["role"] == "student"
]["student_id"].nunique()

if student_count < MIN_STUDENT_COUNT:
    raise ProductionDataMissingError(
        field="cohort_size",
        moodle_function="core_enrol_get_enrolled_users",
        remediation=f"Cohort has {student_count} students (< {MIN_STUDENT_COUNT}). "
                    f"Either enrol more students or lower MIN_STUDENT_COUNT.",
    )

print(f"Real participants: {len(real_participants_long)} rows ({student_count} unique students)")


## 7. Real grade extraction

In [ ]:
grade_summary_rows = []
pairs = real_participants_long[
    real_participants_long["role"] == "student"
][["student_id", "course_id"]].drop_duplicates()

for _, p in pairs.iterrows():
    result = mc.extract_user_grade_items(int(p["student_id"]), int(p["course_id"]))
    if result.get("success"):
        etl_io.save_json(
            result,
            RAW_DIR / f"grades_user_{int(p['student_id'])}_course_{int(p['course_id'])}.json",
        )
    grade_summary_rows.append(
        T.transform_grade_items_summary(result, int(p["student_id"]), int(p["course_id"]))
    )

# Defensive: if no grade extraction happened at all, raise cleanly
if not grade_summary_rows:
    raise ProductionDataMissingError(
        field="grade_items",
        moodle_function="gradereport_user_get_grade_items",
        remediation="No student-course pairs to extract grades for. "
                    "This usually means real_participants_long is empty upstream — "
                    "investigate participant extraction.",
    )

grades_df = pd.DataFrame(grade_summary_rows)
real_grade_rows = int(grades_df["real_grade_available"].fillna(False).sum())
print(f"Real grade items: {real_grade_rows}/{len(grades_df)} pairs")

if real_grade_rows == 0:
    raise ProductionDataMissingError(
        field="grade_items",
        moodle_function="gradereport_user_get_grade_items",
        remediation="No real grade items returned for any student-course pair. "
                    "Verify token has gradereport permission and that grades have been entered in Moodle.",
    )

## 8. Attendance source check (FAIL LOUD)

Attendance has no Moodle source by default — it must come from an external feed (Meshed). Production ETL refuses to emit mocked attendance.

In [ ]:
if not ATTENDANCE_SOURCE_AVAILABLE:
    raise ProductionDataMissingError(
        field="attendance_pct",
        moodle_function="(external — Meshed integration)",
        remediation=(
            "Attendance data is not available from Moodle by default. "
            "Inergrate with the institution's attendance source (Meshed export, mod_attendance plugin, Meshed API), "
            "then set ATTENDANCE_SOURCE_AVAILABLE = True in this notebook."
        ),
    )

# When ATTENDANCE_SOURCE_AVAILABLE is True, replace this stub with the real
# attendance extraction. The DataFrame must have columns:
# student_id, course_id, attendance_pct
# and join cleanly into the cohort.
real_attendance_df = pd.DataFrame()  # TODO: implement extraction here

cohort = real_participants_long.merge(
    grades_df, on=["student_id", "course_id"], how="left"
).merge(
    real_attendance_df, on=["student_id", "course_id"], how="left"
)

# In production, lastaccess from real_participants_long drives days_since_last_login
snapshot = pd.Timestamp.utcnow()
def _resolve_login_days(la):
    if pd.isna(la) or la in (0, "0", None, ""):
        return np.nan  # BLANK -> "Never Login" bin
    try:
        ts = pd.to_datetime(int(la), unit="s", utc=True)
        return int(max(0, (snapshot - ts).days))
    except Exception:
        return np.nan
cohort["days_since_last_login"] = cohort["lastaccess_unix"].apply(_resolve_login_days)

# In production, avg_grade_pct = real value or NaN (NEVER mock)
cohort["avg_grade_pct"] = cohort["avg_grade_pct_real"]
cohort["grade_item_count"] = pd.to_numeric(
    cohort["grade_item_count_real"], errors="coerce"
).fillna(0).astype(int)


## 9. Risk scoring

In [ ]:
cohort_risk = R.apply_final_risk_logic(cohort)
print("Risk level distribution:")
print(cohort_risk["risk_level"].value_counts())


## 10. Build star-schema tables

In [ ]:
fact_student_risk = T.build_fact_student_risk(
    cohort_risk,
    data_source=S.DATA_SOURCE_MOODLE_FULL,
    students_only=True,
)
dim_student = T.build_dim_student(real_participants_long)
dim_course = T.build_dim_course(course_catalog)
dim_site = T.build_dim_site(site_df, base_url_fallback=mc.MOODLE_BASE_URL)
dim_date = T.build_dim_date(fact_student_risk)

print(f"dim_student: {len(dim_student)} rows")
print(f"dim_course: {len(dim_course)} rows")
print(f"fact_student_risk: {len(fact_student_risk)} rows (students only)")


## 11. Assessment-level extension (real only)

In [ ]:
fact_rows = []
dim_rows = []
assignment_dim_rows = []

for cid in course_catalog["course_id"].dropna().astype(int).unique():
    a_result = mc.extract_assignments_for_course(int(cid))
    etl_io.save_json(a_result, RAW_DIR / f"assignments_course_{cid}.json")
    assignment_dim_rows.extend(T.transform_assignments_to_dim_rows(a_result, int(cid)))

students_only = real_participants_long[real_participants_long["role"] == "student"]
pairs = students_only[["student_id", "course_id"]].drop_duplicates()
for _, p in pairs.iterrows():
    g = mc.extract_user_grade_items(int(p["student_id"]), int(p["course_id"]))
    f_rows, d_rows = T.transform_grade_items_to_assessment_rows(
        g, int(p["student_id"]), int(p["course_id"])
    )
    fact_rows.extend(f_rows)
    dim_rows.extend(d_rows)

if not fact_rows:
    raise ProductionDataMissingError(
        field="assessment_rows",
        moodle_function="gradereport_user_get_grade_items / mod_assign_get_assignments",
        remediation="No assessment data returned. Verify Moodle has assessments configured "
                    "and that the token has the relevant permissions.",
    )

fact_assessment_df = pd.DataFrame(fact_rows)
dim_assessment_df = pd.DataFrame(dim_rows + assignment_dim_rows)

fact_assessment = T.build_fact_assessment(fact_assessment_df)
dim_assessment = T.build_dim_assessment(dim_assessment_df)

print(f"dim_assessment: {len(dim_assessment)} rows")
print(f"fact_assessment: {len(fact_assessment)} rows")


## 12. Save outputs

In [ ]:
etl_io.save_dataframe(fact_student_risk, STAR_SCHEMA_DIR / "fact_student_risk")
etl_io.save_dataframe(dim_student, STAR_SCHEMA_DIR / "dim_student")
etl_io.save_dataframe(dim_course, STAR_SCHEMA_DIR / "dim_course")
etl_io.save_dataframe(dim_site, STAR_SCHEMA_DIR / "dim_site")
etl_io.save_dataframe(dim_date, STAR_SCHEMA_DIR / "dim_date")
etl_io.save_dataframe(fact_assessment, STAR_SCHEMA_DIR / "fact_assessment")
etl_io.save_dataframe(dim_assessment, STAR_SCHEMA_DIR / "dim_assessment")

run_summary = pd.DataFrame([{
    "run_timestamp_utc": pd.Timestamp.utcnow().isoformat(),
    "pipeline": "production",
    "site_info_success": True,
    "course_catalog_success": True,
    "participant_extraction_mode": S.DATA_SOURCE_MOODLE_FULL,
    "real_participant_count": int(real_participants_long["student_id"].nunique()),
    "student_count": int((dim_student["role"] == "student").sum()),
    "teacher_count": int(dim_student["role"].isin(S.TEACHER_ROLES).sum()),
    "course_count": int(dim_course["course_id"].nunique()),
    "fact_rows": int(len(fact_student_risk)),
    "real_grade_rows": int(real_grade_rows),
    "risk_low_count": int((fact_student_risk["risk_level"] == "Low").sum()),
    "risk_medium_count": int((fact_student_risk["risk_level"] == "Medium").sum()),
    "risk_high_count": int((fact_student_risk["risk_level"] == "High").sum()),
    "assessment_fact_rows": int(len(fact_assessment)),
    "assessment_count": int(dim_assessment["assessment_id"].nunique()),
    "assessment_data_source": S.DATA_SOURCE_ASSESSMENT_REAL,
    "data_source_summary": f"{S.DATA_SOURCE_MOODLE_FULL}={len(fact_student_risk)}",
    "schema_note": "Production ETL — real-only, no mocks",
}])

etl_io.save_dataframe(run_summary, STAR_SCHEMA_DIR / "etl_run_summary")
display(run_summary.T)
